In [10]:
# ======================================================
# Cell 2 — Install & Import Required Libraries
# ======================================================


import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping
from lightning.pytorch.loggers import CSVLogger
import matplotlib.pyplot as plt

print("✅ Libraries ready!")


✅ Libraries ready!


In [11]:
# ======================================================
# Cell 3 — Prepare Dataset & Dataloaders
# ======================================================
BATCH_SIZE = 32
IMG_SIZE = 224

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

dataset = datasets.ImageFolder(root=DATA_DIR, transform=transform)
class_names = dataset.classes
print("🧠 Classes:", class_names)

# Split into train/val
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_ds, val_ds = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

print(f"✅ Train samples: {len(train_ds)} | Val samples: {len(val_ds)}")


FileNotFoundError: Couldn't find any class folder in /content/data.

In [12]:
import torch
print("torch:", torch.__version__, "| GPU available:", torch.cuda.is_available())

torch: 2.8.0+cu128 | GPU available: True


In [15]:
import os
import math
from pathlib import Path
import random
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import transforms, datasets
import timm
import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping
from lightning.pytorch.loggers import CSVLogger
from torchmetrics.classification import MulticlassAccuracy, MulticlassF1Score

# Path to your dataset root (ImageFolder format: root/class_x/xxx.png)
DATA_DIR = Path('Data')  # <--- update if your dataset folder is named differently
OUT_DIR = Path('outputs')
OUT_DIR.mkdir(exist_ok=True)

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Check device availability
print('torch version:', torch.__version__)
print('GPU available:', torch.cuda.is_available(), ' num GPUs:', torch.cuda.device_count())


torch version: 2.8.0+cu128
GPU available: True  num GPUs: 1


In [16]:
if not DATA_DIR.exists():
    raise FileNotFoundError(f"Dataset path {DATA_DIR} not found. Update DATA_DIR variable.")

classes = [d.name for d in sorted(DATA_DIR.iterdir()) if d.is_dir()]
print('Found classes:', classes)

counts = {c: len(list((DATA_DIR/c).glob('**/*'))) for c in classes}
print('Image counts per class:')
for k, v in counts.items():
    print(k, v)


Found classes: ['Mild Dementia', 'Moderate Dementia', 'Non Demented', 'Very mild Dementia']
Image counts per class:
Mild Dementia 5002
Moderate Dementia 488
Non Demented 67222
Very mild Dementia 13725


In [17]:
IMG_SIZE = 224  # ViT standard size for many pretrained models
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])


In [18]:
print('Image counts per class:')

Image counts per class:


In [19]:
class AlzheimerDataModule(L.LightningDataModule):
    def __init__(self, data_dir, train_transform, val_transform, batch_size=32, num_workers=4):
        super().__init__()
        self.data_dir = Path(data_dir)
        self.train_transform = train_transform
        self.val_transform = val_transform
        self.batch_size = batch_size
        self.num_workers = num_workers

    def setup(self, stage=None):
        full = datasets.ImageFolder(self.data_dir, transform=self.train_transform)
        n = len(full)
        idx = list(range(n))
        random.Random(SEED).shuffle(idx)
        n_train = int(0.8 * n)
        n_val = int(0.1 * n)
        train_idx = idx[:n_train]
        val_idx = idx[n_train:n_train+n_val]
        test_idx = idx[n_train+n_val:]

        from torch.utils.data import Subset
        train_ds = Subset(full, train_idx)
        val_root = datasets.ImageFolder(self.data_dir, transform=self.val_transform)
        val_ds = Subset(val_root, val_idx)
        test_ds = Subset(val_root, test_idx)

        self.train_ds = train_ds
        self.val_ds = val_ds
        self.test_ds = test_ds
        self.num_classes = len(full.classes)
        self.class_to_idx = full.class_to_idx

    def train_dataloader(self):
        return DataLoader(self.train_ds, batch_size=self.batch_size, shuffle=True,
                          num_workers=self.num_workers, pin_memory=True)

    def val_dataloader(self):
        return DataLoader(self.val_ds, batch_size=self.batch_size, shuffle=False,
                          num_workers=self.num_workers, pin_memory=True)

    def test_dataloader(self):
        return DataLoader(self.test_ds, batch_size=self.batch_size, shuffle=False,
                          num_workers=self.num_workers, pin_memory=True)

BATCH_SIZE = 32
NUM_WORKERS = min(8, os.cpu_count() if os.cpu_count() is not None else 4)
data_module = AlzheimerDataModule(DATA_DIR, train_transform, val_transform, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)
data_module.setup()
print('num classes detected:', data_module.num_classes)


num classes detected: 4


In [20]:
class ViTClassifier(L.LightningModule):
    def __init__(self, model_name='vit_base_patch16_224', pretrained=True, num_classes=4, lr=3e-4):
        super().__init__()
        self.save_hyperparameters()
        self.model = timm.create_model(model_name, pretrained=pretrained, num_classes=num_classes)
        self.criterion = nn.CrossEntropyLoss()
        self.accuracy = MulticlassAccuracy(num_classes=num_classes)
        self.f1 = MulticlassF1Score(num_classes=num_classes, average='macro')

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        preds = torch.argmax(logits, dim=1)
        acc = self.accuracy(preds, y)
        self.log('train_loss', loss, on_step=True, on_epoch=True)
        self.log('train_acc', acc, on_step=True, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        preds = torch.argmax(logits, dim=1)
        acc = self.accuracy(preds, y)
        f1 = self.f1(preds, y)
        self.log('val_loss', loss, prog_bar=True, on_epoch=True)
        self.log('val_acc', acc, prog_bar=True, on_epoch=True)
        self.log('val_f1', f1, prog_bar=True, on_epoch=True)

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.hparams.lr, weight_decay=1e-2)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
        return [optimizer], [scheduler]

NUM_CLASSES = data_module.num_classes
model = ViTClassifier(model_name='vit_base_patch16_224', pretrained=True, num_classes=NUM_CLASSES, lr=3e-4) 
print('num classes')


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

num classes


In [21]:
logger = CSVLogger(save_dir='logs', name='alzheimer_vit')
checkpoint_callback = ModelCheckpoint(
    dirpath=OUT_DIR,
    filename='vit-{epoch:02d}-{val_loss:.4f}',
    save_top_k=3,
    monitor='val_loss',
    mode='min'
)
early_stop = EarlyStopping(monitor='val_loss', patience=5, mode='min')

trainer = L.Trainer(
    max_epochs=10,
    callbacks=[checkpoint_callback, early_stop],
    default_root_dir=str(OUT_DIR),
    logger=logger,
    accelerator='auto',
    devices='auto',
    precision=16,
    strategy='auto',
    gradient_clip_val=1.0,
    log_every_n_steps=50,
)
print('num classes')


/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/lightning/fabric/connector.py:571: `precision=16` is supported for historical reasons but its usage is discouraged. Please set your precision to 16-mixed instead!
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


num classes


In [22]:
trainer.fit(model, datamodule=data_module)
print('num classes')

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/lightning/pytorch/utilities/model_summary/model_summary.py:231: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.

  | Name      | Type               | Params | Mode 
---------------------------------------------------------
0 | model     | VisionTransformer  | 85.8 M | train
1 | criterion | CrossEntropyLoss   | 0      | train
2 | accuracy  | MulticlassAccuracy | 0      | train
3 | f1        | MulticlassF1Score  | 0      | train
---------------------------------------------------------
85.8 M    Trainable params
0         Non-trainable params
85.8 M    Total params
343.207   Total estimated model params size (MB)
279       Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=10` reached.


num classes


In [25]:
trainer.test(model, datamodule=data_module)

from PIL import Image
img_path = None  # replace with a real image path, e.g. 'Data/MildDemented/img1.jpg'
if img_path and Path(img_path).exists():
    img = Image.open(img_path).convert('RGB')
    inp = val_transform(img).unsqueeze(0)
    model.eval()
    with torch.no_grad():
        logits = model(inp)
        pred = logits.argmax(dim=1).item()
    print('Predicted class index:', pred)


MisconfigurationException: No `test_step()` method defined to run `Trainer.test`.

In [31]:
class ViTClassifier(L.LightningModule):
    def __init__(self, model_name='vit_base_patch16_224', pretrained=True, num_classes=4, lr=3e-4):
        super().__init__()
        self.save_hyperparameters()
        self.model = timm.create_model(model_name, pretrained=pretrained, num_classes=num_classes)
        self.criterion = nn.CrossEntropyLoss()
        self.accuracy = MulticlassAccuracy(num_classes=num_classes)
        self.f1 = MulticlassF1Score(num_classes=num_classes, average='macro')

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        preds = torch.argmax(logits, dim=1)
        acc = self.accuracy(preds, y)
        self.log('train_loss', loss, on_step=True, on_epoch=True)
        self.log('train_acc', acc, on_step=True, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        preds = torch.argmax(logits, dim=1)
        acc = self.accuracy(preds, y)
        f1 = self.f1(preds, y)
        self.log('val_loss', loss, prog_bar=True, on_epoch=True)
        self.log('val_acc', acc, prog_bar=True, on_epoch=True)
        self.log('val_f1', f1, prog_bar=True, on_epoch=True)

    def test_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        preds = torch.argmax(logits, dim=1)
        acc = self.accuracy(preds, y)
        f1 = self.f1(preds, y)
        self.log('test_loss', loss, prog_bar=True, on_epoch=True)
        self.log('test_acc', acc, prog_bar=True, on_epoch=True)
        self.log('test_f1', f1, prog_bar=True, on_epoch=True)
        return loss

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.hparams.lr, weight_decay=1e-2)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
        return [optimizer], [scheduler]
         
        


In [32]:
NUM_CLASSES = data_module.num_classes
model = ViTClassifier(model_name='vit_base_patch16_224', pretrained=True, num_classes=NUM_CLASSES, lr=3e-4)


In [33]:
trainer.test(model, datamodule=data_module)


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Testing: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.2365237921476364     │
│          test_f1          │    0.02900620736181736    │
│         test_loss         │     2.369785785675049     │
└───────────────────────────┴───────────────────────────┘

[{'test_loss': 2.369785785675049,
  'test_acc': 0.2365237921476364,
  'test_f1': 0.02900620736181736}]

In [34]:
trainer.test(model, datamodule=data_module)

from PIL import Image
img_path = None  # replace with a real image path, e.g. 'Data/MildDemented/img1.jpg'
if img_path and Path(img_path).exists():
    img = Image.open(img_path).convert('RGB')
    inp = val_transform(img).unsqueeze(0)
    model.eval()
    with torch.no_grad():
        logits = model(inp)
        pred = logits.argmax(dim=1).item()
    print('Predicted class index:', pred)


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Testing: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.2365237921476364     │
│          test_f1          │    0.02900620736181736    │
│         test_loss         │     2.369785785675049     │
└───────────────────────────┴───────────────────────────┘

In [35]:
# Cell 1: Import necessary libraries

import os
from typing import Optional
import random
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision.datasets import ImageFolder
from torchvision import transforms
import timm
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping, LearningRateMonitor
from pytorch_lightning.loggers import CSVLogger
import albumentations as A
from albumentations.pytorch import ToTensorV2
import torchmetrics
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR


/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/lightning_fabric/__init__.py:36: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.


In [36]:
# Cell 2: Seed everything for reproducibility

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)


In [37]:
# Cell 3: Define image augmentation functions for training and validation

IMG_SIZE = 224

def get_train_transforms(img_size=IMG_SIZE):
    return A.Compose([
        A.RandomResizedCrop(img_size, img_size, scale=(0.7, 1.0), p=1.0),
        A.HorizontalFlip(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.0625, scale_limit=0.1, rotate_limit=15, p=0.5),
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.02, p=0.5),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])

def get_val_transforms(img_size=IMG_SIZE):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])


In [38]:
# Cell 4: Wrapper Dataset for Albumentations and ImageFolder

class AlbumentationsImageFolder(Dataset):
    def __init__(self, root, transform=None):
        self.dataset = ImageFolder(root)
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        path, label = self.dataset.samples[idx]
        import cv2
        img = cv2.imread(path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.transform:
            img = self.transform(image=img)['image']
        return img, label

    @property
    def classes(self):
        return self.dataset.classes


In [39]:
# Cell 5: Helper functions for MixUp and CutMix

def mixup_data(x, y, alpha=0.4, device='cuda'):
    '''Returns mixed inputs, pairs of targets, and lambda'''
    if alpha <= 0:
        return x, y, None, 1.0
    lam = np.random.beta(alpha, alpha)
    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(device)
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


In [40]:
# Cell 6: Label Smoothing Loss

class LabelSmoothingCrossEntropy(nn.Module):
    def __init__(self, smoothing=0.1):
        super().__init__()
        self.smoothing = smoothing

    def forward(self, preds, target):
        n_class = preds.size(1)
        log_preds = nn.functional.log_softmax(preds, dim=1)
        true_dist = torch.zeros_like(log_preds)
        true_dist.fill_(self.smoothing / (n_class - 1))
        true_dist.scatter_(1, target.data.unsqueeze(1), 1.0 - self.smoothing)
        return torch.mean(torch.sum(-true_dist * log_preds, dim=1))


In [41]:
# Cell 7: Define the ViT Lightning Module

class ViTLightning(pl.LightningModule):
    def __init__(
        self,
        model_name: str = 'vit_base_patch16_224',
        pretrained: bool = True,
        num_classes: int = 2,
        lr: float = 3e-4,
        weight_decay: float = 1e-2,
        mixup_alpha: float = 0.0,
        label_smoothing: float = 0.0,
        class_weights: Optional[torch.Tensor] = None
    ):
        super().__init__()
        self.save_hyperparameters()
        self.model = timm.create_model(model_name, pretrained=pretrained, num_classes=num_classes)
        self.num_classes = num_classes
        self.lr = lr
        self.weight_decay = weight_decay
        self.mixup_alpha = mixup_alpha
        self.label_smoothing = label_smoothing

        # Loss
        if label_smoothing > 0:
            self.criterion = LabelSmoothingCrossEntropy(smoothing=label_smoothing)
        else:
            if class_weights is not None:
                self.criterion = nn.CrossEntropyLoss(weight=class_weights)
            else:
                self.criterion = nn.CrossEntropyLoss()

        # Metrics
        self.train_acc = torchmetrics.Accuracy()
        self.val_acc = torchmetrics.Accuracy()
        self.val_f1 = torchmetrics.F1(num_classes=num_classes, average='macro')

    def forward(self, x):
        return self.model(x)

    def configure_optimizers(self):
        optimizer = AdamW(self.parameters(), lr=self.lr, weight_decay=self.weight_decay)
        scheduler = CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-6)
        return {"optimizer": optimizer, "lr_scheduler": {"scheduler": scheduler, "monitor": "val_loss"}}

    def training_step(self, batch, batch_idx):
        x, y = batch
        # MixUp
        if self.mixup_alpha > 0:
            x, y_a, y_b, lam = mixup_data(x, y, alpha=self.mixup_alpha, device=self.device)
            preds = self(x)
            loss = mixup_criterion(self.criterion, preds, y_a, y_b, lam)
        else:
            preds = self(x)
            loss = self.criterion(preds, y)

        # metrics
        preds_soft = torch.softmax(preds, dim=1)
        preds_labels = torch.argmax(preds_soft, dim=1)
        self.train_acc.update(preds_labels, y)
        self.log("train_loss", loss, prog_bar=False, on_step=True, on_epoch=True)
        self.log("train_acc", self.train_acc, prog_bar=True, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        preds = self(x)
        loss = self.criterion(preds, y)
        preds_labels = torch.argmax(preds, dim=1)
        self.val_acc.update(preds_labels, y)
        self.val_f1.update(preds_labels, y)
        self.log("val_loss", loss, prog_bar=True, on_epoch=True)
        self.log("val_acc", self.val_acc, prog_bar=True, on_epoch=True)
        self.log("val_f1", self.val_f1, prog_bar=True, on_epoch=True)
        return loss


In [42]:
# Cell 8: DataModule for loading and handling datasets

class SimpleDataModule(pl.LightningDataModule):
    def __init__(self, train_dir, val_dir, test_dir=None, batch_size=32, num_workers=4, sampler=None):
        super().__init__()
        self.train_dir = train_dir
        self.val_dir = val_dir
        self.test_dir = test_dir
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.sampler = sampler

    def setup(self, stage=None):
        self.train_ds = AlbumentationsImageFolder(self.train_dir, transform=get_train_transforms())
        self.val_ds = AlbumentationsImageFolder(self.val_dir, transform=get_val_transforms())
        if self.test_dir:
            self.test_ds = AlbumentationsImageFolder(self.test_dir, transform=get_val_transforms())
        self.num_classes = len(self.train_ds.classes)

    def train_dataloader(self):
        if self.sampler is not None:
            return DataLoader(self.train_ds, batch_size=self.batch_size, sampler=self.sampler, num_workers=self.num_workers)
        return DataLoader(self.train_ds, batch_size=self.batch


SyntaxError: incomplete input (3266514373.py, line 23)

In [49]:
# Cell 8: DataModule for loading and handling datasets

class SimpleDataModule(pl.LightningDataModule):
    def __init__(self, train_dir, val_dir, test_dir=None, batch_size=32, num_workers=4, sampler=None):
        super().__init__()
        self.train_dir = train_dir
        self.val_dir = val_dir
        self.test_dir = test_dir
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.sampler = sampler

    def setup(self, stage=None):
        self.train_ds = AlbumentationsImageFolder(self.train_dir, transform=get_train_transforms())
        self.val_ds = AlbumentationsImageFolder(self.val_dir, transform=get_val_transforms())
        if self.test_dir:
            self.test_ds = AlbumentationsImageFolder(self.test_dir, transform=get_val_transforms())
        self.num_classes = len(self.train_ds.classes)

    def train_dataloader(self):
        if self.sampler is not None:
            return DataLoader(self.train_ds, batch_size=self.batch_size, sampler=self.sampler, num_workers=self.num_workers)
        return DataLoader(self.train_ds, batch_size=self.batch_size, shuffle=True, num_workers=self.num_workers)

    def val_dataloader(self):
        return DataLoader(self.val_ds, batch_size=self.batch_size, shuffle=False, num_workers=self.num_workers)

    def test_dataloader(self):
        if self.test_dir:
            return DataLoader(self.test_ds, batch_size=self.batch_size, shuffle=False, num_workers=self.num_workers)
        return None


In [50]:
# Cell 9: Training entrypoint

def train(
    train_dir,
    val_dir,
    test_dir=None,
    model_name='vit_base_patch16_224',
    pretrained=True,
    img_size=IMG_SIZE,
    batch_size=32,
    lr=3e-4,
    weight_decay=1e-2,
    max_epochs=30,
    gpus=1,
    mixup_alpha=0.0,
    label_smoothing=0.0,
):
    # Estimate class weights if imbalance (optional)
    # from collections import Counter
    # counter = Counter([...])  # implement if you need weights
    
    # Initialize the data module with paths
    data_module = SimpleDataModule(train_dir, val_dir, test_dir=test_dir, batch_size=batch_size)
    data_module.setup()
    
    NUM_CLASSES = data_module.num_classes

    # Initialize the model
    model = ViTLightning(
        model_name=model_name,
        pretrained=pretrained,
        num_classes=NUM_CLASSES,
        lr=lr,
        weight_decay=weight_decay,
        mixup_alpha=mixup_alpha,
        label_smoothing=label_smoothing,
    )

    # Define callbacks
    checkpoint_cb = ModelCheckpoint(monitor="val_f1", mode="max", save_top_k=3, filename="{epoch:02d}-{val_f1:.4f}")
    early_stop_cb = EarlyStopping(monitor="val_f1", patience=6, mode="max", strict=False)
    lrmon = LearningRateMonitor(logging_interval='epoch')
    logger = CSVLogger("logs", name="vit_experiment")

    # Trainer setup
    trainer = pl.Trainer(
        max_epochs=max_epochs,
        callbacks=[checkpoint_cb, early_stop_cb, lrmon],
        logger=logger,
        accelerator="gpu" if gpus and torch.cuda.is_available() else "cpu",
        devices=gpus if torch.cuda.is_available() else None,
        precision=16,  # mixed precision
        gradient_clip_val=1.0,
    )

    # Start training
    trainer.fit(model, datamodule=data_module)

    # Test the model if test_dir is provided
    if test_dir:
        trainer.test(model, datamodule=data_module)


In [51]:
# Cell 10: Example Usage

if __name__ == "__main__":
    # Example usage - edit paths & hyperparams
    train(
        train_dir="/data/train",   # Replace with actual path to your training dataset
        val_dir="/data/val",       # Replace with actual path to your validation dataset
        test_dir="/data/test",     # Replace with actual path to your test dataset (optional)
        model_name="vit_base_patch16_224",
        pretrained=True,
        batch_size=32,
        lr=3e-4,
        weight_decay=1e-2,
        max_epochs=30,
        gpus=1,  # Set to the number of GPUs available (use 0 if no GPU)
        mixup_alpha=0.2,
        label_smoothing=0.05,
    )


/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


FileNotFoundError: [Errno 2] No such file or directory: '/data/train'

In [52]:
# Cell 3 (Fixed): Define image augmentation functions for training and validation

IMG_SIZE = 224

def get_train_transforms(img_size=IMG_SIZE):
    return A.Compose([
        A.RandomResizedCrop(size=(img_size, img_size), scale=(0.7, 1.0), p=1.0),  # <-- FIXED
        A.HorizontalFlip(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.0625, scale_limit=0.1, rotate_limit=15, p=0.5),
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.02, p=0.5),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])

def get_val_transforms(img_size=IMG_SIZE):
    return A.Compose([
        A.Resize(height=img_size, width=img_size),  # <-- Make explicit
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])


In [53]:
# Cell 8: DataModule for loading and handling datasets

class SimpleDataModule(pl.LightningDataModule):
    def __init__(self, train_dir, val_dir, test_dir=None, batch_size=32, num_workers=4, sampler=None):
        super().__init__()
        self.train_dir = train_dir
        self.val_dir = val_dir
        self.test_dir = test_dir
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.sampler = sampler

    def setup(self, stage=None):
        # Create datasets using AlbumentationsImageFolder
        self.train_ds = AlbumentationsImageFolder(self.train_dir, transform=get_train_transforms())
        self.val_ds = AlbumentationsImageFolder(self.val_dir, transform=get_val_transforms())
        if self.test_dir:
            self.test_ds = AlbumentationsImageFolder(self.test_dir, transform=get_val_transforms())
        self.num_classes = len(self.train_ds.classes)

    def train_dataloader(self):
        if self.sampler is not None:
            return DataLoader(
                self.train_ds,
                batch_size=self.batch_size,
                sampler=self.sampler,
                num_workers=self.num_workers,
                pin_memory=True,
            )
        return DataLoader(
            self.train_ds,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=self.num_workers,
            pin_memory=True,
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_ds,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers,
            pin_memory=True,
        )

    def test_dataloader(self):
        if self.test_dir:
            return DataLoader(
                self.test_ds,
                batch_size=self.batch_size,
                shuffle=False,
                num_workers=self.num_workers,
                pin_memory=True,
            )
        return None


In [54]:
# Cell 9: Training entrypoint

def train(
    train_dir,
    val_dir,
    test_dir=None,
    model_name='vit_base_patch16_224',
    pretrained=True,
    img_size=IMG_SIZE,
    batch_size=32,
    lr=3e-4,
    weight_decay=1e-2,
    max_epochs=30,
    gpus=1,
    mixup_alpha=0.0,
    label_smoothing=0.0,
):
    # Initialize the DataModule
    data_module = SimpleDataModule(
        train_dir=train_dir,
        val_dir=val_dir,
        test_dir=test_dir,
        batch_size=batch_size,
    )
    data_module.setup()
    NUM_CLASSES = data_module.num_classes

    # Initialize the model
    model = ViTLightning(
        model_name=model_name,
        pretrained=pretrained,
        num_classes=NUM_CLASSES,
        lr=lr,
        weight_decay=weight_decay,
        mixup_alpha=mixup_alpha,
        label_smoothing=label_smoothing,
    )

    # Callbacks
    checkpoint_cb = ModelCheckpoint(
        monitor="val_f1",
        mode="max",
        save_top_k=3,
        filename="{epoch:02d}-{val_f1:.4f}"
    )
    early_stop_cb = EarlyStopping(
        monitor="val_f1",
        patience=6,
        mode="max",
        strict=False
    )
    lrmon = LearningRateMonitor(logging_interval='epoch')
    logger = CSVLogger("logs", name="vit_experiment")

    # Trainer
    trainer = pl.Trainer(
        max_epochs=max_epochs,
        callbacks=[checkpoint_cb, early_stop_cb, lrmon],
        logger=logger,
        accelerator="gpu" if torch.cuda.is_available() and gpus > 0 else "cpu",
        devices=gpus if torch.cuda.is_available() else None,
        precision=16,  # mixed precision
        gradient_clip_val=1.0,
        log_every_n_steps=10,
    )

    # Training
    print(f"🚀 Starting training on {NUM_CLASSES} classes with model {model_name}")
    trainer.fit(model, datamodule=data_module)

    # Optional testing
    if test_dir:
        print("🧪 Running final evaluation on test set...")
        trainer.test(model, datamodule=data_module)


In [55]:
# Cell 10: Example usage

if __name__ == "__main__":
    # ⚙️ Update these paths to your actual dataset locations
    train(
        train_dir="/data/train",   # path to your training images
        val_dir="/data/val",       # path to your validation images
        test_dir="/data/test",     # optional test set path
        model_name="vit_base_patch16_224",  # ViT model from timm
        pretrained=True,
        batch_size=32,
        lr=3e-4,
        weight_decay=1e-2,
        max_epochs=30,
        gpus=1,                    # set 0 if CPU only
        mixup_alpha=0.2,
        label_smoothing=0.05,
    )


FileNotFoundError: [Errno 2] No such file or directory: '/data/train'